<a href="https://colab.research.google.com/github/elviakiran-miranda-hue/BUS4118S26/blob/dev/Prompt_ENG_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

class ProfessionalTravelAgent:
    def __init__(self):
        self.context = {"name": None, "email": None, "booking_id": None}
        self.escalated = False
        # Constraint: Critical keywords that trigger immediate human handoff
        self.critical_triggers = ["lawyer", "sue", "legal", "police", "emergency", "manager"]

    def sanitize(self, text):
        # Constraint: Prevent storage of SSNs (9 digits)
        if re.search(r'\d{3}-\d{2}-\d{4}|\d{9}', text):
            return "[REDACTED_SENSITIVE_DATA]"
        return text

    def internal_react_cycle(self, user_input):
        """
        Hidden ReACT Cycle:
        - THOUGHT: Analyze user intent (Booking vs. Complaint).
        - ACTION: Update context or request missing Booking ID.
        - OBSERVATION: Determine if a solution is possible or if escalation is required.
        """
        clean_input = self.sanitize(user_input)
        if clean_input == "[REDACTED_SENSITIVE_DATA]":
            return "Safety Alert: We noticed sensitive data. Please provide only your name and booking ID for help."

        # 1. IMMEDIATE ACTION: Escalation Check
        if any(word in clean_input.lower() for word in self.critical_triggers):
            self.escalated = True
            return "I recognize this is a serious matter. I am escalating this conversation to a senior manager immediately."

        # 2. GATHER IDENTITY (The Foundation)
        email_match = re.search(r'\S+@\S+', clean_input)
        if email_match: self.context['email'] = email_match.group(0)

        if not self.context['name']:
            if "name is" in clean_input.lower():
                self.context['name'] = clean_input.lower().split("is")[-1].strip().title()
            elif len(clean_input.split()) <= 3 and "@" not in clean_input:
                self.context['name'] = clean_input.strip().title()

        if not self.context['name'] or not self.context['email']:
            return "Welcome to TravelCo! To assist you, please provide your full name and email address."

        # 3. COMPLAINT & ISSUE HANDLING
        input_lower = clean_input.lower()
        booking_match = re.search(r'BK-\d{4}', clean_input.upper())
        if booking_match: self.context['booking_id'] = booking_match.group(0)

        # Handle Cancellations and Changes
        if any(word in input_lower for word in ["cancel", "change", "complaint", "wrong", "issue"]):
            if not self.context['booking_id']:
                return f"I understand you'd like to make a change or cancellation, {self.context['name']}. Please provide your Booking Number (BK-XXXX)."

            if "cancel" in input_lower:
                return f"I've processed the cancellation for {self.context['booking_id']}. A confirmation email will be sent to {self.context['email']}."
            if "change" in input_lower:
                return f"I've opened the modification portal for {self.context['booking_id']}. What new dates or locations were you considering?"

            return f"I see your complaint regarding {self.context['booking_id']}. I've logged this for review. Would you like to speak to a supervisor, or should I try to rebook you?"

        # 4. BOOKING FLOW (Package/Flight/Hotel)
        if any(word in input_lower for word in ["package", "flight", "hotel", "book"]):
            return f"Excellent, {self.context['name']}. I'm pulling up our current availability for that. Where is your dream destination?"

        return f"Hello {self.context['name']}, how can I help you today? I can assist with new bookings or help you change/cancel an existing trip."

    def start_chat(self):
        print("--- Travel Support AI Initialized (Complaints & Escalations Enabled) ---")
        while not self.escalated:
            try:
                user_msg = input("User: ")
                if not user_msg.strip(): continue

                response = self.internal_react_cycle(user_msg)
                print(f"AI: {response}")

            except EOFError:
                break

if __name__ == "__main__":
    bot = ProfessionalTravelAgent()
    bot.start_chat()

--- Travel Support AI Initialized (Complaints & Escalations Enabled) ---
